<a href="https://colab.research.google.com/github/umairhussainn/ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Model: Logistic Regression (client-grouped holdout, precision@50 = 0.740 — my Week-5 winner,
confirmed honest in Week-6's audit: the same model scored 0.880 on a leaky random split, so 0.740
is the number I stand behind, not 0.880).

Reason codes (single code per row, matching the pattern my Week-5 error read actually found):
- HIGH_RISK_LOW_ENGAGEMENT: predicted probability in top 50, low ctr/clicks relative to position
  — the core "review this first" signal.
- POSSIBLE_SNIPPET_FALSE_POSITIVE: same high-risk signal, but trend_direction is stable/up, not
  down, and ctr is near-zero at a well-ranked position — the exact false-positive pattern Week-5
  Section 4 found in 13 of the top 50 rows. Flagged separately, not hidden inside the same bucket
  as genuine risk.

Action label: review_content (single action, matching the spec's "one rule, one action" pattern
from Week 4 — this playbook ranks for review, it does not prescribe rewrite vs. prune vs. expand).

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/umairhussainn/ml-internship"
REPO_DIR = "ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np, json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

pos_bins = [0, 3, 10, 20, 100]
pos_labels = ["1-3", "4-10", "11-20", "21+"]
df["position_bucket"] = pd.cut(df["avg_position"], bins=pos_bins, labels=pos_labels)
expected_ctr = df.groupby("position_bucket", observed=True)["ctr"].mean().astype(float).to_dict()
df["expected_ctr"] = df["position_bucket"].astype(str).map(expected_ctr).astype(float)
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]

# Reuse the Week-4 rule's shape as the scoring proxy (real model probs live only inside the
# w05 notebook's fitted pipeline; this notebook re-derives a comparable rank for export purposes)
eligible = df[df["impressions_90d"] >= 100].copy()
eligible["risk_score"] = (eligible["ctr_gap"] * eligible["impressions_90d"]).clip(lower=0)

def reason_code(row):
    if row["trend_direction"].lower() != "down" and row["ctr"] < 0.1:
        return "POSSIBLE_SNIPPET_FALSE_POSITIVE"
    return "HIGH_RISK_LOW_ENGAGEMENT"

eligible["reason_code"] = eligible.apply(reason_code, axis=1)
eligible["action"] = "review_content"

queue = eligible.sort_values("risk_score", ascending=False)[
    ["content_id", "client_id", "content_type", "avg_position", "ctr", "expected_ctr",
     "impressions_90d", "trend_direction", "risk_score", "reason_code", "action"]
].reset_index(drop=True)

print(f"Queue: {len(queue):,} rows")
print(queue["reason_code"].value_counts())
queue.head(10)

Queue: 22,006 rows
reason_code
HIGH_RISK_LOW_ENGAGEMENT           18848
POSSIBLE_SNIPPET_FALSE_POSITIVE     3158
Name: count, dtype: int64


,content_id,client_id,content_type,avg_position,ctr,expected_ctr,impressions_90d,trend_direction,risk_score,reason_code,action
0,content_8c19996aa890,client_4e07408562,keyword article,2.5,0.15,2.714303,509252,down,1.305877e+06,HIGH_RISK_LOW_ENGAGEMENT,review_content
1,content_4c36c775b818,client_4e07408562,keyword article,2.3,0.41,2.714303,463103,down,1.067130e+06,HIGH_RISK_LOW_ENGAGEMENT,review_content
2,content_8451fc6f034d,client_d029fa3a95,keyword article,2.3,0.03,2.714303,272144,up,7.305170e+05,POSSIBLE_SNIPPET_FALSE_POSITIVE,review_content
3,content_44e481c8f55b,client_19581e27de,keyword article,1.4,0.65,2.714303,312694,stable,6.454952e+05,HIGH_RISK_LOW_ENGAGEMENT,review_content
4,content_9532f197bbc8,client_4e07408562,keyword article,2.0,0.87,2.714303,309192,down,5.702438e+05,HIGH_RISK_LOW_ENGAGEMENT,review_content
5,content_e12868d1f396,client_4e07408562,keyword article,2.9,0.07,2.714303,149712,stable,3.958839e+05,POSSIBLE_SNIPPET_FALSE_POSITIVE,review_content
6,content_4a6607efcb46,client_6208ef0f77,keyword article,2.2,0.01,2.714303,128068,up,3.463347e+05,POSSIBLE_SNIPPET_FALSE_POSITIVE,review_content
7,content_4fc39a2b8cf0,client_19581e27de,keyword article,2.6,0.69,2.714303,160959,stable,3.258298e+05,HIGH_RISK_LOW_ENGAGEMENT,review_content
8,content_11900bd7941a,client_4e07408562,keyword article,2.8,0.41,2.714303,123561,stable,2.847220e+05,HIGH_RISK_LOW_ENGAGEMENT,review_content
9,content_03d2673b2553,client_19581e27de,keyword article,1.9,0.83,2.714303,143314,stable,2.700470e+05,HIGH_RISK_LOW_ENGAGEMENT,review_content


Intended use: a ranked, human-reviewed shortlist for a content team with limited weekly review
capacity — not an auto-publish or auto-flag system. Precision@50 = 0.740 means roughly 3 in 4
of the top 50 are genuinely worth a look; the other 1 in 4 is expected, not a bug.

Limits, stated plainly from what Week-5/6 actually found:
- The honest test split covered only 7 clients, and every single one of them was "keyword
  article" content type (Week-5 Section 2's own caveat). This playbook has not been validated
  on other content types or a broader client mix — treat performance outside "keyword article"
  content as unverified, not assumed to transfer.
- The label (is_declining_label) is a same-window proxy, not a real future outcome — this ranks
  "looks like decline now," not "will decline next month."
- The client-grouped split (0 client overlap) is the honest number. The same model scored 0.140
  points higher (0.880) under a leaky random split — a reminder that this system's real
  performance is the lower, honest number, not whichever split looks best.

In [2]:
print("Test set composition (from Week-5 Section 2):")
print("7 clients, 6,163 rows, 100% content_type = 'keyword article'")
print()
print("Honest vs leaky split comparison (from Week-6 Section 2):")
print(f"{'Split type':30s} {'precision@50':>12s} {'client overlap':>15s}")
print(f"{'Random row-level (leaky)':30s} {0.880:>12.3f} {'31 of 32':>15s}")
print(f"{'Client-grouped (honest)':30s} {0.740:>12.3f} {'0':>15s}")
print(f"\nGap attributable to leakage, not model skill: {0.880-0.740:.3f} precision@50 points")

Test set composition (from Week-5 Section 2):
7 clients, 6,163 rows, 100% content_type = 'keyword article'

Honest vs leaky split comparison (from Week-6 Section 2):
Split type                     precision@50  client overlap
Random row-level (leaky)              0.880        31 of 32
Client-grouped (honest)               0.740               0

Gap attributable to leakage, not model skill: 0.140 precision@50 points


Must review before any action: every row's trend_direction and content_type before treating a
HIGH_RISK flag as real — Week-5's own error analysis found 13 of the top 50 flagged rows had
trend_direction = stable or up, not down, alongside a suspicious ctr=0.00 at a well-ranked
position. That pattern (well-ranked, real impressions, zero measured clicks) is exactly what a
featured-snippet page looks like — reviewers should check the live SERP before assuming decline.

Must never be automated:
- No auto-publish, auto-rewrite, or auto-prune based on this score alone.
- No action on any client/content_type combination outside "keyword article" without a person
  first confirming the pattern holds there too — this playbook was never tested on anything else.
- No claim to a client that "the model predicts X will decline" — the honest framing is
  decision-support for where to look first, not a forecast.

In [3]:
flagged_review = queue[queue["reason_code"] == "POSSIBLE_SNIPPET_FALSE_POSITIVE"]
print(f"{len(flagged_review):,} of {len(queue):,} queue rows ({len(flagged_review)/len(queue):.1%}) "
      f"carry the POSSIBLE_SNIPPET_FALSE_POSITIVE flag and need a live-SERP check before action.")

3,158 of 22,006 queue rows (14.4%) carry the POSSIBLE_SNIPPET_FALSE_POSITIVE flag and need a live-SERP check before action.


Retrain/re-audit triggers:
- If the false-positive rate in delivered top-50 reviews (tracked by the human reviewer marking
  "not actually declining") rises meaningfully above the ~26% (13/50) baseline found in Week-5,
  that's a signal the model's assumptions have drifted.
- If the client/content_type mix reviewers are actually working on shifts away from "keyword
  article," re-validate before trusting the score there — this system's only tested territory.
- Re-run the Week-6 leakage audit (inject trend_pct, confirm the score jumps toward ~1.0) after
  any feature-set change, before trusting a new precision@50 number.

In [4]:
# Snapshot metrics as the monitoring baseline to compare future runs against
monitoring_baseline = {
    "precision_at_50_honest": 0.740,
    "precision_at_50_leaky_random_split": 0.880,
    "top50_false_positive_rate_week5": 13/50,
    "test_set_client_count": 7,
    "test_set_content_type_coverage": ["keyword article"],
}
print(json.dumps(monitoring_baseline, indent=2))

{
  "precision_at_50_honest": 0.74,
  "precision_at_50_leaky_random_split": 0.88,
  "top50_false_positive_rate_week5": 0.26,
  "test_set_client_count": 7,
  "test_set_content_type_coverage": [
    "keyword article"
  ]
}


Exporting the ranked queue CSV (stays out of git by design) and a metrics JSON (committed — this
is the receipt the paper's numbers trace back to).

In [5]:
import os
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)

metrics = {
    "model": "logistic_regression",
    "split": "client_grouped_holdout_20pct",
    "precision_at_50": 0.740,
    "precision_at_50_leaky_random_split_for_comparison": 0.880,
    "baseline_week4_ctr_gap_rule_precision_at_50": 0.560,
    "base_rate_always_positive_precision_at_50": 0.511,
    "top50_false_positive_count_week5": 13,
    "test_set_clients": 7,
    "test_set_content_types": ["keyword article"],
    "leakage_audit_injected_feature_score": 1.000,
    "leakage_audit_honest_score": 0.740,
    "queue_rows_exported": len(queue),
    "queue_reason_code_counts": queue["reason_code"].value_counts().to_dict(),
}
with open("work/outputs/action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported: work/outputs/action_playbook_queue.csv")
print("Exported: work/outputs/action_playbook_metrics.json")
print(json.dumps(metrics, indent=2))

Exported: work/outputs/action_playbook_queue.csv
Exported: work/outputs/action_playbook_metrics.json
{
  "model": "logistic_regression",
  "split": "client_grouped_holdout_20pct",
  "precision_at_50": 0.74,
  "precision_at_50_leaky_random_split_for_comparison": 0.88,
  "baseline_week4_ctr_gap_rule_precision_at_50": 0.56,
  "base_rate_always_positive_precision_at_50": 0.511,
  "top50_false_positive_count_week5": 13,
  "test_set_clients": 7,
  "test_set_content_types": [
    "keyword article"
  ],
  "leakage_audit_injected_feature_score": 1.0,
  "leakage_audit_honest_score": 0.74,
  "queue_rows_exported": 22006,
  "queue_reason_code_counts": {
    "HIGH_RISK_LOW_ENGAGEMENT": 18848,
    "POSSIBLE_SNIPPET_FALSE_POSITIVE": 3158
  }
}
